# Held-out clinical-field validation for direct Oλ
Evaluate the TCAT GEP and atlas hard-label cell-type geometries separately. For every omitted dataset, use its seven-dataset midpoint transition memberships and the corresponding seven-dataset continuous clinical coefficients. Compute low-Oλ (adverse) minus high-Oλ (favorable) organization within the omitted dataset. The held-out outcomes are not inputs to the field calculations.

Run all cells once the survival notebook writes its eight-fold membership CSV. The independent single-dataset fits and pooled clinical-field run are separate. Each eligible fold uses 500 dataset-adjusted correlation draws, arithmetic-mean matrices, 5,000 within-cohort membership permutations, and 10,000 clinical-field surrogates per family. The final section repeats specificity testing with the CLR and feature-scale sensitivity from notebook 80.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import time
import numpy as np
import pandas as pd
from IPython.display import display
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
# Set only if multiple matching survival runs exist.
LODO_RUN = None

In [ ]:
TCAT_PARTICLES = DATA_DIR / 'patient_TCAT_particles' / 'all_patient_TCAT_particles_harmonized.npz'
TCAT_HAZARDS = DATA_DIR / 'tcat_survival_crossfit_adjusted_min1000_v1' / 'crossfit_feature_results.csv'
ATLAS_PARTICLES = (DATA_DIR / 'Functional_CART_atlas_patient_particles_hard_labels_all8' /
                   'patient_atlas_particles.npz')
ATLAS_HAZARDS = (DATA_DIR / 'atlas_cell_type_survival_crossfit_adjusted_min1000_v1' /
                 'crossfit_feature_results.csv')

ANALYSES = {
    'TCAT_GEP': {
        'particles': TCAT_PARTICLES, 'hazards': TCAT_HAZARDS, 'prefix': 'TCAT_GEP_',
        'feature_transform': ('cell-level Bayesian-bootstrap-weighted mean of positive TCAT GEP compositions; '
                              'CLR_j = log(x_j) - mean_k(log(x_k)); no GEP pseudocount added'),
        'compositional_constraint': 'sum of TCAT GEP CLR coordinates equals zero before correlation standardization',
    },
    'atlas_cell_type': {
        'particles': ATLAS_PARTICLES, 'hazards': ATLAS_HAZARDS, 'prefix': 'ATLAS_proportion_',
        'feature_transform': ('cell-level Bayesian-bootstrap-weighted hard-label counts multiplied by n_cells; '
                              'add 0.5 to every cell-type count, normalize to proportions, then '
                              'CLR_j = log(p_j) - mean_k(log(p_k))'),
        'compositional_constraint': 'sum of atlas cell-type CLR coordinates equals zero before correlation standardization',
    },
}

QUICK_TEST = False
if QUICK_TEST:
    CORRELATION_DRAWS, MEMBERSHIP_PERMUTATIONS, FIELD_SURROGATES = 40, 100, 200
else:
    CORRELATION_DRAWS, MEMBERSHIP_PERMUTATIONS, FIELD_SURROGATES = 500, 5000, 10000
MINIMUM_EFFECTIVE_REGIME = 3.0
RANDOM_STATE = 0
OVERWRITE = False


def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

stage=DATA_DIR/'olambda_reciprocal_direct_no_asymptote_floor_v1'
estimates=stage/'patient_olambda_asymptotes.csv'
first_inputs=[estimates,stage/'patient_olambda_particles.csv',DATA_DIR/'clinical_outcomes.csv']
hashes={p.name:digest(p) for p in first_inputs}
keys=['Dataset','sample_id']
cohort=pd.read_csv(estimates).set_index(keys,verify_integrity=True)
assert len(cohort)==163 and cohort.index.get_level_values(0).nunique()==8
assert cohort.olambda_scale.eq('reciprocal').all() and cohort.model.eq('bounded_direct_no_asymptote_floor').all()
parent=DATA_DIR/'bayesian_Olambda_survival'/'direct_reciprocal_adjusted_midpoint_v1'/'leave_one_dataset_out'
candidates=[Path(LODO_RUN)] if LODO_RUN else sorted(parent.glob('*/notebook_config.json'))
matches=[]
for candidate in candidates:
    run=candidate if LODO_RUN else candidate.parent
    config=json.loads((run/'notebook_config.json').read_text())
    if (config['inputs']==hashes and config['adjustments']=={'Treatment':'axicel','Disease':'lymphoma'}
            and (run/'crossfit_patient_regime_probabilities.csv').exists()):
        matches.append(run)
if len(matches)!=1:
    raise ValueError(f'Expected one completed eight-fold direct membership export under {parent}. '
                     f'Matches: {matches}. Finish the survival fold export or select its LODO_RUN.')
run=matches[0]
CROSSFIT_MEMBERSHIP=run/'crossfit_patient_regime_probabilities.csv'
membership_table=pd.read_csv(CROSSFIT_MEMBERSHIP)
ordered=membership_table.set_index(keys,verify_integrity=True).sort_index()
assert ordered.index.equals(cohort.index.sort_values())
assert membership_table.Dataset.eq(membership_table.heldout_dataset).all()
assert ordered.posterior_mean_probability_high.between(0,1).all()
if 'posterior_mean_probability_adverse' in ordered:
    np.testing.assert_allclose(ordered.posterior_mean_probability_adverse,1-ordered.posterior_mean_probability_high,atol=1e-12)
support=[]
for d,g in membership_table.groupby('Dataset'):
    folder=run/'training_fits'/('heldout_'+d)
    meta=json.loads((folder/'smooth_transition_Q10_Q90'/'run_manifest.json').read_text())
    assert meta['hazard_activation_offset']==0.5 and meta['transition_scale_sd']==0.1
    assert meta['adjustments']=={'Treatment':'axicel','Disease':'lymphoma'}
    training=pd.read_csv(folder/'training_outcomes.csv')
    included=cohort.index.intersection(pd.MultiIndex.from_frame(training[keys]))
    assert set(included.get_level_values(0))==set(membership_table.Dataset)-{d}
    assert len(included)==len(cohort)-len(g)
    q=g.posterior_mean_probability_high.to_numpy()
    low=1-q
    support.append(dict(heldout_dataset=d,patients=len(q),low_adverse_mass=low.sum(),
                        high_favorable_mass=q.sum(),eligible=min(q.sum(),low.sum())>=MINIMUM_EFFECTIVE_REGIME,
                        low_membership_ess=low.sum()**2/(low@low) if low.sum() else 0.,
                        high_membership_ess=q.sum()**2/(q@q) if q.sum() else 0.))
support=pd.DataFrame(support)
display(support)
if not support.eligible.any(): raise ValueError('No cohort meets the regime support threshold.')
signature=dict(version=1,run=str(run),membership=hashlib.sha256(ordered.posterior_mean_probability_high.to_numpy(dtype='<f8').tobytes()).hexdigest(),
               files={str(p):digest(p) for p in [TCAT_PARTICLES,TCAT_HAZARDS,ATLAS_PARTICLES,ATLAS_HAZARDS]},
               draws=CORRELATION_DRAWS,permutations=MEMBERSHIP_PERMUTATIONS,surrogates=FIELD_SURROGATES,
               minimum_mass=MINIMUM_EFFECTIVE_REGIME,seed=RANDOM_STATE)
tag=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()[:12]
OUTPUT_DIR=DATA_DIR/'predictive_field_fully_crossfit_direct_midpoint_mean_v1'/tag
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
(OUTPUT_DIR/'input_manifest.json').write_text(json.dumps(signature,indent=2))
support.to_csv(OUTPUT_DIR/'regime_support_all_folds.csv',index=False)
print('Using memberships:',CROSSFIT_MEMBERSHIP,'\nOutput:',OUTPUT_DIR)

In [ ]:
METRICS = ['prognostic_channeling', 'signed_graph_variation', 'signed_neighbor_cosine']
DESIRED_SIGN = np.asarray([1.0, -1.0, 1.0])
PSD_TOLERANCE = -1e-10


def safe_name(value):
    return ''.join(character if character.isalnum() or character in '._-' else '_' for character in str(value))


def weighted_correlation_and_scale(values, weights):
    weights = np.asarray(weights, dtype=float)
    total = weights.sum()
    if not np.isfinite(total) or total <= 0:
        raise ValueError('Correlation weights have no positive finite mass.')
    weights = weights / total
    centered = values - np.sum(weights[:, None] * values, axis=0)
    covariance = (centered * weights[:, None]).T @ centered
    scale = np.sqrt(np.maximum(np.diag(covariance), 1e-12))
    correlation = covariance / scale[:, None] / scale[None, :]
    correlation = (correlation + correlation.T) / 2
    np.fill_diagonal(correlation, 1)
    return correlation, scale


def weighted_correlation(values, weights):
    return weighted_correlation_and_scale(values, weights)[0]


def dataset_residuals(values, weights, datasets):
    residuals = np.asarray(values, dtype=float).copy()
    for dataset in np.unique(datasets):
        keep = datasets == dataset
        group_weights = weights[keep]
        group_weights = group_weights / group_weights.sum()
        residuals[keep] -= np.sum(values[keep] * group_weights[:, None], axis=0)
    return residuals


def posterior_states(values, particle_weights, datasets, draws, random_state):
    rng = np.random.default_rng(random_state)
    patients, particles, features = values.shape
    adjusted = np.empty((draws, patients, features), dtype=np.float32)
    bootstrap_weights = np.empty((draws, patients), dtype=np.float32)
    for draw in range(draws):
        particle_index = np.asarray([
            rng.choice(particles, p=particle_weights[patient])
            for patient in range(patients)
        ])
        weights = rng.dirichlet(np.ones(patients))
        state = values[np.arange(patients), particle_index]
        adjusted[draw] = dataset_residuals(state, weights, datasets)
        bootstrap_weights[draw] = weights
    return adjusted, bootstrap_weights


def mean_correlations(states, bootstrap_weights, membership, indices, feature_indices=None,
                      return_scales=False):
    if feature_indices is None:
        feature_indices = np.arange(states.shape[2])
    adverse_sum = np.zeros((len(feature_indices), len(feature_indices)), dtype=float)
    favorable_sum = np.zeros_like(adverse_sum)
    adverse_scales = np.empty((states.shape[0], len(feature_indices)), dtype=np.float32)
    favorable_scales = np.empty_like(adverse_scales)
    membership = np.asarray(membership, dtype=float)
    for draw in range(states.shape[0]):
        values = states[draw, indices][:, feature_indices]
        weights = bootstrap_weights[draw, indices]
        adverse_correlation, adverse_scale = weighted_correlation_and_scale(
            values, weights * membership
        )
        favorable_correlation, favorable_scale = weighted_correlation_and_scale(
            values, weights * (1 - membership)
        )
        adverse_sum += adverse_correlation
        favorable_sum += favorable_correlation
        adverse_scales[draw] = adverse_scale
        favorable_scales[draw] = favorable_scale
    adverse = adverse_sum / states.shape[0]
    favorable = favorable_sum / states.shape[0]
    if return_scales:
        return adverse, favorable, adverse_scales, favorable_scales
    return adverse, favorable


def save_reproduction_inputs(path, states, bootstrap_weights, datasets, sample_ids,
                             features, metadata):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        dataset_adjusted_state_draws=np.asarray(states, dtype=np.float32),
        bayesian_bootstrap_weights=np.asarray(bootstrap_weights, dtype=np.float32),
        datasets=np.asarray(datasets, dtype='U'),
        sample_ids=np.asarray(sample_ids, dtype='U'),
        feature_names=np.asarray(features, dtype='U'),
        metadata_json=np.asarray(json.dumps(metadata, sort_keys=True)),
    )


def save_field_information(path, adverse, favorable, adverse_scales, favorable_scales,
                           beta, features, raw_membership, adverse_membership,
                           patient_indices, removed_coordinates, reproduction_inputs,
                           coefficient_centers, coefficient_scales, adverse_high, metadata):
    path.parent.mkdir(parents=True, exist_ok=True)
    high = adverse if adverse_high else favorable
    low = favorable if adverse_high else adverse
    high_scales = adverse_scales if adverse_high else favorable_scales
    low_scales = favorable_scales if adverse_high else adverse_scales
    np.savez_compressed(
        path,
        high_mean_correlation=np.asarray(high, dtype=float),
        low_mean_correlation=np.asarray(low, dtype=float),
        adverse_mean_correlation=np.asarray(adverse, dtype=float),
        favorable_mean_correlation=np.asarray(favorable, dtype=float),
        high_feature_sd_by_draw=np.asarray(high_scales, dtype=np.float32),
        low_feature_sd_by_draw=np.asarray(low_scales, dtype=np.float32),
        adverse_feature_sd_by_draw=np.asarray(adverse_scales, dtype=np.float32),
        favorable_feature_sd_by_draw=np.asarray(favorable_scales, dtype=np.float32),
        clinical_beta=np.asarray(beta, dtype=float),
        feature_names=np.asarray(features, dtype='U'),
        raw_probability_high=np.asarray(raw_membership, dtype=float),
        adverse_regime_membership=np.asarray(adverse_membership, dtype=float),
        patient_indices=np.asarray(patient_indices, dtype=int),
        removed_coordinates=np.asarray(removed_coordinates, dtype='U'),
        coefficient_feature_centers=np.asarray(coefficient_centers, dtype=float),
        coefficient_feature_scales=np.asarray(coefficient_scales, dtype=float),
        adverse_high_on_original_predictor=np.asarray(bool(adverse_high)),
        reproduction_inputs_path=np.asarray(str(reproduction_inputs)),
        metadata_json=np.asarray(json.dumps(metadata, sort_keys=True)),
    )


def save_psd_failure_bundle(path, states, bootstrap_weights, membership, indices,
                            feature_indices, adverse_mean, favorable_mean):
    """Retain the actual input matrices if a mean unexpectedly fails PSD."""
    if feature_indices is None:
        feature_indices = np.arange(states.shape[2])
    adverse_draws = np.empty(
        (states.shape[0], len(feature_indices), len(feature_indices)), dtype=np.float32
    )
    favorable_draws = np.empty_like(adverse_draws)
    for draw in range(states.shape[0]):
        values = states[draw, indices][:, feature_indices]
        weights = bootstrap_weights[draw, indices]
        adverse_draws[draw] = weighted_correlation(values, weights * membership)
        favorable_draws[draw] = weighted_correlation(values, weights * (1 - membership))
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path, adverse_draws=adverse_draws, favorable_draws=favorable_draws,
        adverse_mean=adverse_mean, favorable_mean=favorable_mean,
        membership=np.asarray(membership, dtype=float),
    )


def field_metrics(correlation, beta):
    beta = np.asarray(beta, dtype=float)
    first, second = np.triu_indices(len(beta), 1)
    edge = correlation[first, second]
    weight = edge ** 2
    sign = np.where(edge < 0, -1.0, 1.0)
    degree = np.zeros(len(beta))
    np.add.at(degree, first, weight)
    np.add.at(degree, second, weight)
    adjacency = np.zeros_like(correlation)
    adjacency[first, second] = weight * sign
    adjacency[second, first] = weight * sign
    neighbor = adjacency @ beta / np.maximum(degree, 1e-12)
    beta_norm = np.maximum(beta @ beta, 1e-12)
    variation_norm = np.maximum(np.sum(degree * beta ** 2), 1e-12)
    cosine_norm = np.sqrt(
        np.maximum(np.sum(degree * beta ** 2), 1e-12)
        * np.maximum(np.sum(degree * neighbor ** 2), 1e-12)
    )
    return np.asarray([
        beta @ correlation @ beta / beta_norm,
        np.sum(weight * (beta[first] - sign * beta[second]) ** 2) / variation_norm,
        np.sum(degree * beta * neighbor) / cosine_norm,
    ])


def field_contrast(adverse, favorable, beta):
    return field_metrics(adverse, beta) - field_metrics(favorable, beta)


def permute_within_dataset(values, groups, rng):
    permuted = np.asarray(values, dtype=float).copy()
    for group in groups:
        permuted[group] = permuted[rng.permutation(group)]
    return permuted


def permutation_summary(observed, null):
    null_mean = null.mean(axis=0)
    null_sd = np.maximum(null.std(axis=0, ddof=1), 1e-12)
    z = (observed - null_mean) / null_sd
    p = (1 + np.sum(null * DESIRED_SIGN >= observed * DESIRED_SIGN, axis=0)) / (len(null) + 1)
    oriented_z = z * DESIRED_SIGN
    joint_z = oriented_z.mean()
    joint_null = ((null - null_mean) / null_sd * DESIRED_SIGN).mean(axis=1)
    joint_p = (1 + np.sum(joint_null >= joint_z)) / (len(null) + 1)
    return z, p, joint_z, joint_p, joint_null


def matrix_diagnostics(matrix):
    matrix = np.asarray(matrix, dtype=float)
    symmetry_error = float(np.max(np.abs(matrix - matrix.T)))
    diagonal_deviation = float(np.max(np.abs(np.diag(matrix) - 1)))
    eigenvalues = np.linalg.eigvalsh((matrix + matrix.T) / 2)
    return np.asarray([
        eigenvalues.min(), np.sum(eigenvalues < PSD_TOLERANCE),
        symmetry_error, diagonal_deviation,
    ], dtype=float)


def diagnostic_record(scope, representation, context, regime, matrix, **extra):
    minimum, negative, symmetry, diagonal = matrix_diagnostics(matrix)
    return {
        'scope': scope, 'representation': representation, 'context': context,
        'regime': regime, 'minimum_eigenvalue': minimum,
        'eigenvalues_below_minus_1e_minus_10': int(negative),
        'maximum_symmetry_error': symmetry,
        'maximum_diagonal_deviation': diagonal, **extra,
    }


def save_permutation_diagnostics(path, diagnostics):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        adverse_minimum_eigenvalue=diagnostics[:, 0, 0],
        adverse_negative_eigenvalues=diagnostics[:, 0, 1].astype(int),
        adverse_symmetry_error=diagnostics[:, 0, 2],
        adverse_diagonal_deviation=diagnostics[:, 0, 3],
        favorable_minimum_eigenvalue=diagnostics[:, 1, 0],
        favorable_negative_eigenvalues=diagnostics[:, 1, 1].astype(int),
        favorable_symmetry_error=diagnostics[:, 1, 2],
        favorable_diagonal_deviation=diagnostics[:, 1, 3],
    )


def diagnostic_summary(scope, representation, context, diagnostics, **extra):
    return {
        'scope': scope, 'representation': representation, 'context': context,
        'permutations': len(diagnostics),
        'minimum_eigenvalue_across_permutations': float(diagnostics[:, :, 0].min()),
        'matrices_failing_psd_tolerance': int(np.sum(diagnostics[:, :, 1] > 0)),
        'maximum_negative_eigenvalue_count': int(diagnostics[:, :, 1].max()),
        'maximum_symmetry_error': float(diagnostics[:, :, 2].max()),
        'maximum_diagonal_deviation': float(diagnostics[:, :, 3].max()),
        **extra,
    }


def benjamini_hochberg(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values)
    adjusted = values[order] * len(values) / np.arange(1, len(values) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    answer = np.empty(len(values), dtype=float)
    answer[order] = np.minimum(adjusted, 1)
    return answer

In [ ]:
SURROGATE_FAMILIES = ['permuted_field', 'sign_randomized_field', 'norm_matched_isotropic_field']


def surrogate_vector(beta, family, rng):
    if family == 'permuted_field':
        return beta[rng.permutation(len(beta))]
    if family == 'sign_randomized_field':
        return np.abs(beta) * rng.choice([-1.0, 1.0], size=len(beta))
    if family == 'norm_matched_isotropic_field':
        vector = rng.normal(size=len(beta))
        return np.linalg.norm(beta) * vector / np.linalg.norm(vector)
    raise ValueError(family)


def combine_folds(folds, null_lookup, probability_column='permutation_p'):
    rows = []
    standardized = []
    for fold in folds.itertuples(index=False):
        null = null_lookup[fold.heldout_dataset]
        null_mean = null.mean(axis=0)
        null_sd = np.maximum(null.std(axis=0, ddof=1), 1e-12)
        standardized.append((null - null_mean) / null_sd * DESIRED_SIGN)
    standardized = np.stack(standardized)
    for metric_index, metric in enumerate(METRICS):
        observed = np.mean(folds[f'z__{metric}'].to_numpy(float) * DESIRED_SIGN[metric_index])
        null = standardized[:, :, metric_index].mean(axis=0)
        rows.append({
            'metric': metric, 'datasets': len(folds), 'mean_oriented_z': observed,
            probability_column: (1 + np.sum(null >= observed)) / (len(null) + 1),
            'datasets_in_expected_direction': int(np.sum(
                folds[f'contrast__{metric}'].to_numpy(float) * DESIRED_SIGN[metric_index] > 0
            )),
        })
    observed = folds.joint_oriented_z.mean()
    null = standardized.mean(axis=(0, 2))
    rows.append({
        'metric': 'three_metric_joint', 'datasets': len(folds),
        'mean_oriented_z': observed,
        probability_column: (1 + np.sum(null >= observed)) / (len(null) + 1),
        'datasets_in_expected_direction': int(np.sum(folds.joint_oriented_z > 0)),
    })
    return pd.DataFrame(rows)

## Membership permutations and unconstrained surrogate fields
Positive oriented statistics indicate stronger clinical-field organization in the adverse, low-Oλ regime. The joint statistic averages the three oriented z scores. Combined tests weight eligible datasets equally.

In [ ]:
all_fold_results = []
all_combined_results = []
all_observed_diagnostics = []
all_permutation_diagnostics = []
all_surrogate_folds = []
all_surrogate_combined = []

for representation_number, (representation, specification) in enumerate(ANALYSES.items()):
    print('\n' + '#' * 110)
    print('FULLY HELD-OUT REPRESENTATION:', representation)
    print('#' * 110, flush=True)
    archive = np.load(specification['particles'], allow_pickle=False)
    all_features = archive['feature_names'].astype(str)
    features = np.asarray([feature for feature in all_features if feature.startswith(specification['prefix'])])
    lookup = {feature: index for index, feature in enumerate(all_features)}
    feature_indices = np.asarray([lookup[feature] for feature in features])
    patients = pd.DataFrame({
        'Dataset': archive['datasets'].astype(str), 'sample_id': archive['patient_ids'].astype(str),
        'particle_row': np.arange(len(archive['patient_ids'])),
    }).merge(
        membership_table[['Dataset', 'sample_id', 'heldout_dataset', 'posterior_mean_probability_high']],
        on=['Dataset', 'sample_id'], how='inner', validate='one_to_one',
    ).sort_values(['Dataset', 'sample_id']).reset_index(drop=True)
    assert pd.MultiIndex.from_frame(patients[keys]).equals(ordered.index)
    assert len(features)==len(set(features))==(52 if representation=="TCAT_GEP" else 11)
    rows = patients.particle_row.to_numpy(int)
    values = np.asarray(archive['values'][rows][:, :, feature_indices], dtype=float)
    particle_weights = np.asarray(archive['particle_weights'][rows], dtype=float)
    particle_weights /= particle_weights.sum(axis=1, keepdims=True)
    datasets = patients.Dataset.astype(str).to_numpy()
    states, bootstrap_weights = posterior_states(
        values, particle_weights, datasets, CORRELATION_DRAWS, RANDOM_STATE
    )
    reproduction_inputs = OUTPUT_DIR / 'correlation_reproduction_inputs' / f'{representation}.npz'
    save_reproduction_inputs(
        reproduction_inputs, states, bootstrap_weights, datasets,
        patients.sample_id.astype(str).to_numpy(), features,
        {
            'representation': representation,
            'feature_transform': specification['feature_transform'],
            'compositional_constraint': specification['compositional_constraint'],
            'dataset_adjustment': 'draw-specific Bayesian-bootstrap weighted dataset-mean subtraction',
            'correlation_scaling': 'draw- and regime-specific weighted population SD',
            'correlation_draws': CORRELATION_DRAWS,
            'random_state': RANDOM_STATE,
            'draw_specific_correlation_matrices_saved': False,
            'reproduction': 'use saved state draws, bootstrap weights, held-out indices, fold membership, and feature order',
        },
    )
    hazards = pd.read_csv(specification['hazards'])

    representation_dir = OUTPUT_DIR / representation
    matrix_dir = representation_dir / 'observed_mean_matrices'
    null_dir = representation_dir / 'membership_permutation_nulls'
    diagnostic_dir = representation_dir / 'permutation_psd_diagnostics'
    surrogate_dir = representation_dir / 'surrogate_field_nulls'
    bundle_dir = representation_dir / 'compositional_sensitivity_bundles'
    for directory in [representation_dir, matrix_dir, null_dir, diagnostic_dir, surrogate_dir, bundle_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    fold_rows = []
    fold_null = {}
    surrogate_rows = []
    surrogate_null_lookup = {family: {} for family in SURROGATE_FAMILIES}
    for fold_number, heldout in enumerate(sorted(patients.Dataset.unique()), start=1):
        checkpoint_path = representation_dir / f'checkpoint__{safe_name(heldout)}.json'
        bundle_path = bundle_dir / f'{safe_name(heldout)}__O_lambda.npz'
        index = np.flatnonzero(datasets == heldout)
        probability_high = patients.loc[index, 'posterior_mean_probability_high'].to_numpy(float)
        adverse_membership = 1 - probability_high
        effective_high = probability_high.sum()
        effective_low = len(index) - effective_high
        if min(effective_high, effective_low) < MINIMUM_EFFECTIVE_REGIME:
            print(f'SKIP {heldout}: {effective_low:.2f} low, {effective_high:.2f} high', flush=True)
            continue
        if checkpoint_path.exists() and bundle_path.exists() and not OVERWRITE:
            checkpoint = json.loads(checkpoint_path.read_text())
            fold_rows.append(checkpoint['fold_result'])
            all_observed_diagnostics.extend(checkpoint['observed_diagnostics'])
            all_permutation_diagnostics.append(checkpoint['permutation_diagnostic_summary'])
            null_archive = np.load(null_dir / f'{safe_name(heldout)}.npz')
            fold_null[heldout] = null_archive['null']
            for surrogate_row in checkpoint['surrogate_results']:
                surrogate_rows.append(surrogate_row)
            for family in SURROGATE_FAMILIES:
                surrogate_archive = np.load(surrogate_dir / f'{family}__{safe_name(heldout)}.npz')
                surrogate_null_lookup[family][heldout] = surrogate_archive['null']
            print(f'[{fold_number}/{patients.Dataset.nunique()}] RESUME {heldout}', flush=True)
            continue

        effects = hazards.loc[hazards.heldout_dataset.astype(str).eq(heldout)].set_index('feature',verify_integrity=True)
        missing = sorted(set(features).difference(effects.index.astype(str)))
        if missing:
            raise ValueError(f'{heldout} is missing {len(missing)} cross-fitted effects.')
        beta = np.log(effects.loc[features, 'HR_median'].to_numpy(float))
        assert np.isfinite(beta).all() and np.isfinite(effects.loc[features,'scale']).all() and effects.loc[features,'scale'].gt(0).all()
        adverse, favorable, adverse_scales, favorable_scales = mean_correlations(
            states, bootstrap_weights, adverse_membership, index,
            return_scales=True,
        )
        observed = field_contrast(adverse, favorable, beta)
        observed_diagnostics = [
            diagnostic_record('heldout_observed', representation, 'O_lambda', 'adverse', adverse,
                              heldout_dataset=heldout),
            diagnostic_record('heldout_observed', representation, 'O_lambda', 'favorable', favorable,
                              heldout_dataset=heldout),
        ]
        for diagnostic in observed_diagnostics:
            if diagnostic['eigenvalues_below_minus_1e_minus_10']:
                failure_path = (
                    representation_dir / 'psd_failures' /
                    f'{safe_name(heldout)}__observed.npz'
                )
                save_psd_failure_bundle(
                    failure_path, states, bootstrap_weights, adverse_membership,
                    index, None, adverse, favorable,
                )
                raise RuntimeError(f'Mean matrix failed PSD audit; inputs saved to {failure_path}: {diagnostic}')

        information_metadata = {
            'scope': 'fully_heldout', 'heldout_dataset': heldout,
            'representation': representation, 'context': 'O_lambda',
            'geometry': 'dataset_adjusted within held-out dataset',
            'aggregation': 'arithmetic mean of draw-specific correlation matrices',
            'feature_transform': specification['feature_transform'],
            'compositional_constraint': specification['compositional_constraint'],
            'dataset_adjustment': 'draw-specific Bayesian-bootstrap weighted dataset-mean subtraction',
            'correlation_scaling': 'weighted population SD calculated separately for every draw and regime',
            'clinical_coefficients': 'posterior-median held-out-fold log HR per global feature SD',
            'clinical_vector_centering': 'none', 'clinical_vector_projection': 'none',
            'surrogate_centering': 'none', 'surrogate_projection': 'none',
            'draw_specific_correlation_matrices_saved': False,
            'reproduction_inputs_saved': True,
            'random_state': RANDOM_STATE,
            'membership_permutation_seed': RANDOM_STATE + fold_number,
            'membership_permutation_rule': 'permute cross-fitted adverse (low-Oλ) membership within the held-out dataset',
            'surrogate_seed_rule': 'random_state + 1000 * fold_number + 100 * family_index',
        }
        save_field_information(
            bundle_path, adverse, favorable, adverse_scales, favorable_scales,
            beta, features, probability_high, adverse_membership, index, [],
            reproduction_inputs,
            effects.loc[features, 'center'].to_numpy(float) if 'center' in effects else np.full(len(features), np.nan),
            effects.loc[features, 'scale'].to_numpy(float) if 'scale' in effects else np.full(len(features), np.nan),
            False,
            information_metadata,
        )

        pd.DataFrame(adverse, index=features, columns=features).to_csv(
            matrix_dir / f'{safe_name(heldout)}__adverse.csv'
        )
        pd.DataFrame(favorable, index=features, columns=features).to_csv(
            matrix_dir / f'{safe_name(heldout)}__favorable.csv'
        )

        rng = np.random.default_rng(RANDOM_STATE + fold_number)
        null = np.empty((MEMBERSHIP_PERMUTATIONS, len(METRICS)), dtype=float)
        permutation_diagnostics = np.empty((MEMBERSHIP_PERMUTATIONS, 2, 4), dtype=float)
        group = [np.arange(len(index))]
        partial = checkpoint_path.with_suffix('.partial.npz')
        start = 0
        if partial.exists() and not OVERWRITE:
            with np.load(partial) as saved:
                start = len(saved['null'])
                null[:start] = saved['null']
                permutation_diagnostics[:start] = saved['diagnostics']
                rng.bit_generator.state = json.loads(str(saved['rng']))
            print('Resuming fold at permutation',start,flush=True)
        for permutation in range(start,MEMBERSHIP_PERMUTATIONS):
            permuted = permute_within_dataset(adverse_membership, group, rng)
            permuted_adverse, permuted_favorable = mean_correlations(
                states, bootstrap_weights, permuted, index
            )
            null[permutation] = field_contrast(permuted_adverse, permuted_favorable, beta)
            permutation_diagnostics[permutation, 0] = matrix_diagnostics(permuted_adverse)
            permutation_diagnostics[permutation, 1] = matrix_diagnostics(permuted_favorable)
            if np.any(permutation_diagnostics[permutation, :, 1] > 0):
                failure_path = (
                    representation_dir / 'psd_failures' /
                    f'{safe_name(heldout)}__permutation_{permutation + 1}.npz'
                )
                save_psd_failure_bundle(
                    failure_path, states, bootstrap_weights, permuted,
                    index, None, permuted_adverse, permuted_favorable,
                )
                raise RuntimeError(
                    f'Permutation mean matrix failed PSD audit; inputs saved to {failure_path}'
                )
            if (permutation + 1) % 100 == 0:
                tmp = partial.with_suffix('.tmp.npz')
                np.savez(tmp,null=null[:permutation+1],diagnostics=permutation_diagnostics[:permutation+1],
                         rng=json.dumps(rng.bit_generator.state))
                tmp.replace(partial)
            if (permutation + 1) % max(1, MEMBERSHIP_PERMUTATIONS // 10) == 0:
                print(f'    membership permutations {permutation + 1:,}/{MEMBERSHIP_PERMUTATIONS:,}', flush=True)
        z, p, joint_z, joint_p, joint_null = permutation_summary(observed, null)
        fold_null[heldout] = null
        np.savez_compressed(null_dir / f'{safe_name(heldout)}.npz', null=null, joint_null=joint_null)
        save_permutation_diagnostics(diagnostic_dir / f'{safe_name(heldout)}.npz', permutation_diagnostics)
        diagnostic_summary_row = diagnostic_summary(
            'heldout_membership_permutations', representation, 'O_lambda', permutation_diagnostics,
            heldout_dataset=heldout,
        )
        fold_row = {
            'representation': representation, 'heldout_dataset': heldout,
            'patients': len(index), 'effective_high_patients': effective_high,
            'effective_low_patients': effective_low, 'aggregation': 'arithmetic_mean',
            'joint_oriented_z': joint_z, 'joint_permutation_p': joint_p,
        }
        for metric_index, metric in enumerate(METRICS):
            fold_row[f'contrast__{metric}'] = observed[metric_index]
            fold_row[f'z__{metric}'] = z[metric_index]
            fold_row[f'p__{metric}'] = p[metric_index]

        fold_surrogate_rows = []
        for family_index, family in enumerate(SURROGATE_FAMILIES):
            surrogate_rng = np.random.default_rng(
                RANDOM_STATE + 1000 * fold_number + 100 * family_index
            )
            surrogate_null = np.empty((FIELD_SURROGATES, len(METRICS)), dtype=float)
            for surrogate in range(FIELD_SURROGATES):
                surrogate_null[surrogate] = field_contrast(
                    adverse, favorable, surrogate_vector(beta, family, surrogate_rng)
                )
            surrogate_z, surrogate_p, surrogate_joint_z, surrogate_joint_p, surrogate_joint_null = permutation_summary(
                observed, surrogate_null
            )
            surrogate_null_lookup[family][heldout] = surrogate_null
            np.savez_compressed(
                surrogate_dir / f'{family}__{safe_name(heldout)}.npz',
                null=surrogate_null, joint_null=surrogate_joint_null,
            )
            surrogate_row = {
                'representation': representation, 'surrogate_family': family,
                'heldout_dataset': heldout, 'patients': len(index),
                'effective_high_patients': effective_high, 'effective_low_patients': effective_low,
                'joint_oriented_z': surrogate_joint_z, 'joint_surrogate_p': surrogate_joint_p,
            }
            for metric_index, metric in enumerate(METRICS):
                surrogate_row[f'contrast__{metric}'] = observed[metric_index]
                surrogate_row[f'z__{metric}'] = surrogate_z[metric_index]
                surrogate_row[f'p__{metric}'] = surrogate_p[metric_index]
            fold_surrogate_rows.append(surrogate_row)
            surrogate_rows.append(surrogate_row)

        checkpoint = {
            'fold_result': fold_row, 'observed_diagnostics': observed_diagnostics,
            'permutation_diagnostic_summary': diagnostic_summary_row,
            'surrogate_results': fold_surrogate_rows,
        }
        tmp = checkpoint_path.with_suffix('.tmp.json')
        tmp.write_text(json.dumps(checkpoint,indent=2))
        tmp.replace(checkpoint_path)
        partial.unlink(missing_ok=True)
        fold_rows.append(fold_row)
        all_observed_diagnostics.extend(observed_diagnostics)
        all_permutation_diagnostics.append(diagnostic_summary_row)
        print(
            f'[{fold_number}/{patients.Dataset.nunique()}] {heldout} | '
            f'joint z={joint_z:.2f}, p={joint_p:.4g}', flush=True
        )

    folds = pd.DataFrame(fold_rows)
    folds.to_csv(representation_dir / 'mean_crossfit_predictive_field_by_dataset.csv', index=False)
    combined = combine_folds(folds, fold_null, 'permutation_p')
    combined.insert(0, 'representation', representation)
    combined.to_csv(representation_dir / 'mean_crossfit_predictive_field_combined.csv', index=False)
    all_fold_results.append(folds)
    all_combined_results.append(combined)

    surrogate_folds = pd.DataFrame(surrogate_rows)
    surrogate_folds.to_csv(representation_dir / 'mean_random_field_by_dataset.csv', index=False)
    combined_families = []
    for family in SURROGATE_FAMILIES:
        family_folds = surrogate_folds.loc[surrogate_folds.surrogate_family.eq(family)].copy()
        family_combined = combine_folds(family_folds, surrogate_null_lookup[family], 'surrogate_p')
        family_combined.insert(0, 'surrogate_family', family)
        family_combined.insert(0, 'representation', representation)
        combined_families.append(family_combined)
    surrogate_combined = pd.concat(combined_families, ignore_index=True)
    surrogate_combined.to_csv(representation_dir / 'mean_random_field_combined.csv', index=False)
    all_surrogate_folds.append(surrogate_folds)
    all_surrogate_combined.append(surrogate_combined)

pd.concat(all_fold_results, ignore_index=True).to_csv(
    OUTPUT_DIR / 'mean_crossfit_predictive_field_by_dataset.csv', index=False
)
pd.concat(all_combined_results, ignore_index=True).to_csv(
    OUTPUT_DIR / 'mean_crossfit_predictive_field_combined.csv', index=False
)
pd.concat(all_surrogate_folds, ignore_index=True).to_csv(
    OUTPUT_DIR / 'mean_random_field_by_dataset.csv', index=False
)
pd.concat(all_surrogate_combined, ignore_index=True).to_csv(
    OUTPUT_DIR / 'mean_random_field_combined.csv', index=False
)
pd.DataFrame(all_observed_diagnostics).to_csv(
    OUTPUT_DIR / 'observed_mean_matrix_psd_diagnostics.csv', index=False
)
pd.DataFrame(all_permutation_diagnostics).to_csv(
    OUTPUT_DIR / 'permutation_mean_matrix_psd_summary.csv', index=False
)
bundle_inventory = []
for path in sorted(OUTPUT_DIR.glob('*/compositional_sensitivity_bundles/*.npz')):
    archive = np.load(path, allow_pickle=False)
    metadata = json.loads(str(archive['metadata_json']))
    bundle_inventory.append({
        **metadata, 'bundle_path': str(path),
        'features': len(archive['feature_names']),
        'correlation_draws': archive['adverse_feature_sd_by_draw'].shape[0],
        'removed_coordinates': ';'.join(archive['removed_coordinates'].astype(str)),
        'reproduction_inputs_path': str(archive['reproduction_inputs_path']),
    })
pd.DataFrame(bundle_inventory).to_csv(
    OUTPUT_DIR / 'compositional_sensitivity_bundle_inventory.csv', index=False
)
print('Saved:', OUTPUT_DIR)

## Composition-constrained specificity
For both complete CLR representations, use the saved fold-specific coefficient scales and regime-specific scale draws for the observed field and the surrogate fields. Report all three families by dataset and in combination, alongside the unconstrained results.

In [ ]:
CROSSFIT_INPUT_DIR = OUTPUT_DIR
CROSSFIT_OUTPUT_DIR = OUTPUT_DIR / 'compositional_sensitivity'
CROSSFIT_OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
def safe_name(value):
    return ''.join(
        character if character.isalnum() or character in '._-' else '_'
        for character in str(value)
    )


def field_metric_matrices(correlation):
    correlation = np.asarray(correlation, dtype=float)
    first, second = np.triu_indices(len(correlation), 1)
    edge = correlation[first, second]
    weight = edge ** 2
    sign = np.where(edge < 0, -1.0, 1.0)

    degree = np.zeros(len(correlation), dtype=float)
    np.add.at(degree, first, weight)
    np.add.at(degree, second, weight)

    adjacency = np.zeros_like(correlation)
    adjacency[first, second] = weight * sign
    adjacency[second, first] = weight * sign
    laplacian = np.diag(degree) - adjacency
    return correlation, adjacency, laplacian, degree


def field_metrics_batch(correlation, fields):
    fields = np.atleast_2d(np.asarray(fields, dtype=float))
    correlation, adjacency, laplacian, degree = field_metric_matrices(correlation)

    field_norm = np.maximum(np.sum(fields ** 2, axis=1), 1e-12)
    channeling = np.sum((fields @ correlation) * fields, axis=1) / field_norm

    degree_energy = np.maximum(np.sum(fields ** 2 * degree[None, :], axis=1), 1e-12)
    variation = np.sum((fields @ laplacian) * fields, axis=1) / degree_energy

    neighbor = (fields @ adjacency) / np.maximum(degree[None, :], 1e-12)
    neighbor_energy = np.maximum(
        np.sum(neighbor ** 2 * degree[None, :], axis=1), 1e-12
    )
    cosine = (
        np.sum(fields * neighbor * degree[None, :], axis=1)
        / np.sqrt(degree_energy * neighbor_energy)
    )
    return np.column_stack([channeling, variation, cosine])


def mean_tangent_map(feature_sd_by_draw):
    feature_sd_by_draw = np.asarray(feature_sd_by_draw, dtype=float)
    dimensions = feature_sd_by_draw.shape[1]
    answer = np.zeros((dimensions, dimensions), dtype=float)
    identity = np.eye(dimensions)
    for scale in feature_sd_by_draw:
        if not np.isfinite(scale).all() or np.any(scale <= 0):
            raise ValueError('Feature-SD vectors must be positive and finite.')
        direction = scale / np.linalg.norm(scale)
        projection = identity - np.outer(direction, direction)
        answer += projection @ np.diag(scale)
    return answer / len(feature_sd_by_draw)


def canonical_raw_field(beta, coefficient_scales):
    beta = np.asarray(beta, dtype=float)
    coefficient_scales = np.asarray(coefficient_scales, dtype=float)
    if not np.isfinite(coefficient_scales).all() or np.any(coefficient_scales <= 0):
        raise ValueError('Coefficient feature scales are missing or invalid.')
    raw = beta / coefficient_scales
    raw -= raw.mean()
    if np.linalg.norm(raw) <= 1e-12:
        raise ValueError('The projected clinical field has zero norm.')
    return raw


def raw_field_without_constraint(beta, coefficient_scales):
    beta = np.asarray(beta, dtype=float)
    coefficient_scales = np.asarray(coefficient_scales, dtype=float)
    if not np.isfinite(coefficient_scales).all() or np.any(coefficient_scales <= 0):
        raise ValueError('Coefficient feature scales are missing or invalid.')
    raw = beta / coefficient_scales
    if np.linalg.norm(raw) <= 1e-12:
        raise ValueError('The clinical field has zero norm.')
    return raw


def mean_standardization_map(feature_sd_by_draw):
    feature_sd_by_draw = np.asarray(feature_sd_by_draw, dtype=float)
    if not np.isfinite(feature_sd_by_draw).all() or np.any(feature_sd_by_draw <= 0):
        raise ValueError('Feature-SD vectors must be positive and finite.')
    return np.diag(feature_sd_by_draw.mean(axis=0))


def normalize_rows(values, target_norm):
    values = np.asarray(values, dtype=float)
    norm = np.linalg.norm(values, axis=1)
    if np.any(norm <= 1e-12):
        raise ValueError('At least one surrogate has zero norm after projection.')
    return values * (target_norm / norm)[:, None]


def surrogate_matrix(base_field, family, rng, count, compositional):
    base_field = np.asarray(base_field, dtype=float)
    dimensions = len(base_field)
    target_norm = np.linalg.norm(base_field)

    if family == 'permuted_field':
        order = np.argsort(rng.random((count, dimensions)), axis=1)
        values = base_field[order]
    elif family == 'sign_randomized_field':
        values = np.abs(base_field)[None, :] * rng.choice(
            [-1.0, 1.0], size=(count, dimensions)
        )
    elif family == 'norm_matched_isotropic_field':
        values = rng.normal(size=(count, dimensions))
    else:
        raise ValueError(family)

    if compositional:
        values -= values.mean(axis=1, keepdims=True)
    return normalize_rows(values, target_norm)


def surrogate_summary(observed, null):
    null = np.asarray(null, dtype=float)
    null_mean = null.mean(axis=0)
    null_sd = np.maximum(null.std(axis=0, ddof=1), 1e-12)
    z = (observed - null_mean) / null_sd
    p = (
        1 + np.sum(null * DESIRED_SIGN >= observed * DESIRED_SIGN, axis=0)
    ) / (len(null) + 1)

    oriented_z = z * DESIRED_SIGN
    joint_z = oriented_z.mean()
    joint_null = ((null - null_mean) / null_sd * DESIRED_SIGN).mean(axis=1)
    joint_p = (1 + np.sum(joint_null >= joint_z)) / (len(null) + 1)
    return z, p, joint_z, joint_p, joint_null


def benjamini_hochberg(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values)
    adjusted = values[order] * len(values) / np.arange(1, len(values) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    answer = np.empty(len(values), dtype=float)
    answer[order] = np.minimum(adjusted, 1)
    return answer

In [ ]:
def projection_diagnostic(scale_by_draw, raw_field):
    residuals = []
    fractions_removed = []
    directions = []
    for scale in np.asarray(scale_by_draw, dtype=float):
        direction = scale / np.linalg.norm(scale)
        unprojected = scale * raw_field
        projected = unprojected - direction * (direction @ unprojected)
        residuals.append(
            abs(direction @ projected) / max(np.linalg.norm(projected), 1e-12)
        )
        fractions_removed.append(
            np.linalg.norm(unprojected - projected)
            / max(np.linalg.norm(unprojected), 1e-12)
        )
        directions.append(direction)
    directions = np.asarray(directions)
    representative = directions.mean(axis=0)
    representative /= np.linalg.norm(representative)
    cosines = directions @ representative
    return {
        'maximum_constraint_residual': float(np.max(residuals)),
        'median_fraction_field_norm_removed': float(np.median(fractions_removed)),
        'maximum_fraction_field_norm_removed': float(np.max(fractions_removed)),
        'minimum_constraint_direction_cosine': float(np.min(cosines)),
        'q025_constraint_direction_cosine': float(np.quantile(cosines, 0.025)),
    }


def load_bundle(path):
    archive = np.load(path, allow_pickle=False)
    metadata = json.loads(str(archive['metadata_json']))
    return archive, metadata


def evaluate_bundle(path, scope, bundle_number, output_directory, fold_number=None):
    archive, metadata = load_bundle(path)
    representation = str(metadata['representation'])
    context = str(metadata['context'])
    heldout = str(metadata.get('heldout_dataset', ''))
    feature_names = set(archive['feature_names'].astype(str))
    removed = archive['removed_coordinates'].astype(str)
    source_features = len(feature_names) + len(removed)
    if representation == 'TCAT_GEP' and context == 'TCAT_SCORE_ASA_CLR':
        retained_components = sorted(ASA_COMPONENT_GEPS.intersection(feature_names))
        if retained_components:
            raise ValueError(
                'The saved ASA bundle still contains its component GEPs: '
                + ', '.join(retained_components)
                + '. Rerun notebook 77 with the corrected exclusion rule before this sensitivity.'
            )
        if set(removed) != ASA_COMPONENT_GEPS:
            raise ValueError(
                'The ASA bundle must record exactly its four component GEPs as removed; found: '
                + ', '.join(sorted(removed))
            )
        if source_features != 52 or len(feature_names) != 48:
            raise ValueError(
                f'Expected the publication ASA geometry to change from 52 to 48 GEPs; '
                f'found {source_features} to {len(feature_names)}.'
            )
        print('    ASA geometry audit: 52 source GEPs -> 48 field nodes', flush=True)
    constraint_applied = len(removed) == 0

    beta = np.asarray(archive['clinical_beta'], dtype=float)
    coefficient_scales = np.asarray(archive['coefficient_feature_scales'], dtype=float)
    adverse_scales = np.asarray(archive['adverse_feature_sd_by_draw'], dtype=float)
    favorable_scales = np.asarray(archive['favorable_feature_sd_by_draw'], dtype=float)
    adverse_correlation = np.asarray(archive['adverse_mean_correlation'], dtype=float)
    favorable_correlation = np.asarray(archive['favorable_mean_correlation'], dtype=float)

    diagnostics = {
        'scope': scope,
        'representation': representation,
        'context': context,
        'heldout_dataset': heldout,
        'source_features': source_features,
        'features': len(beta),
        'removed_coordinates': ';'.join(removed),
        'constraint_applied': constraint_applied,
        'projection_method': (
            'mean of draw-specific standardized CLR tangent maps'
            if constraint_applied else
            'not applicable after removal of one CLR coordinate'
        ),
    }

    if constraint_applied:
        raw_field = canonical_raw_field(beta, coefficient_scales)
        adverse_map = mean_tangent_map(adverse_scales)
        favorable_map = mean_tangent_map(favorable_scales)
        adverse_field = adverse_map @ raw_field
        favorable_field = favorable_map @ raw_field
        for regime, scale in [('adverse', adverse_scales), ('favorable', favorable_scales)]:
            for key, value in projection_diagnostic(scale, raw_field).items():
                diagnostics[f'{regime}_{key}'] = value
    else:
        raw_field = raw_field_without_constraint(beta, coefficient_scales)
        adverse_map = mean_standardization_map(adverse_scales)
        favorable_map = mean_standardization_map(favorable_scales)
        adverse_field = adverse_map @ raw_field
        favorable_field = favorable_map @ raw_field

    observed = (
        field_metrics_batch(adverse_correlation, adverse_field)[0]
        - field_metrics_batch(favorable_correlation, favorable_field)[0]
    )

    result_rows = []
    nulls = {}
    for family_index, family in enumerate(SURROGATE_FAMILIES):
        if scope == 'fully_crossfit':
            seed = RANDOM_STATE + 1000 * int(fold_number) + 100 * family_index
        else:
            seed = RANDOM_STATE + 100000 + 1000 * bundle_number + 100 * family_index
        rng = np.random.default_rng(seed)
        raw_surrogates = surrogate_matrix(
            raw_field, family, rng, FIELD_SURROGATES, constraint_applied
        )
        adverse_surrogates = raw_surrogates @ adverse_map.T
        favorable_surrogates = raw_surrogates @ favorable_map.T
        null = (
            field_metrics_batch(adverse_correlation, adverse_surrogates)
            - field_metrics_batch(favorable_correlation, favorable_surrogates)
        )
        z, p, joint_z, joint_p, joint_null = surrogate_summary(observed, null)
        nulls[family] = null

        row = {
            'scope': scope,
            'representation': representation,
            'context': context,
            'heldout_dataset': heldout,
            'surrogate_family': family,
            'features': len(beta),
            'removed_coordinates': ';'.join(removed),
            'constraint_applied': constraint_applied,
            'surrogates': FIELD_SURROGATES,
            'random_state': seed,
            'joint_oriented_z': float(joint_z),
            'joint_surrogate_p': float(joint_p),
        }
        for metric_number, metric in enumerate(METRICS):
            row[f'contrast__{metric}'] = float(observed[metric_number])
            row[f'z__{metric}'] = float(z[metric_number])
            row[f'p__{metric}'] = float(p[metric_number])
        result_rows.append(row)

    return result_rows, diagnostics, nulls


def run_collection(bundle_paths, scope, output_directory):
    checkpoint_directory = output_directory / 'checkpoints'
    null_directory = output_directory / 'surrogate_nulls'
    checkpoint_directory.mkdir(parents=True, exist_ok=True)
    null_directory.mkdir(parents=True, exist_ok=True)

    paths = list(bundle_paths)
    fold_names = sorted({
        json.loads(str(np.load(path, allow_pickle=False)['metadata_json'])).get(
            'heldout_dataset', ''
        )
        for path in paths
    })
    fold_lookup = {name: number for number, name in enumerate(fold_names, start=1)}

    all_rows = []
    all_diagnostics = []
    null_lookup = {}
    for bundle_number, path in enumerate(paths, start=1):
        archive, metadata = load_bundle(path)
        representation = str(metadata['representation'])
        context = str(metadata['context'])
        heldout = str(metadata.get('heldout_dataset', ''))
        stem = '__'.join(filter(None, [representation, heldout, context]))
        stem = safe_name(stem)
        checkpoint = checkpoint_directory / f'{stem}.json'
        expected_nulls = {
            family: null_directory / f'{stem}__{family}.npz'
            for family in SURROGATE_FAMILIES
        }

        resume_is_current = checkpoint.exists() and all(
            path.exists() for path in expected_nulls.values()
        )
        if (
            resume_is_current
            and representation == 'TCAT_GEP'
            and context == 'TCAT_SCORE_ASA_CLR'
        ):
            bundle_features = set(archive['feature_names'].astype(str))
            bundle_removed = set(archive['removed_coordinates'].astype(str))
            saved_checkpoint = json.loads(checkpoint.read_text())
            saved_diagnostic = saved_checkpoint.get('diagnostic', {})
            resume_is_current = (
                len(bundle_features) == 48
                and bundle_removed == ASA_COMPONENT_GEPS
                and int(saved_diagnostic.get('features', -1)) == 48
                and set(filter(None, str(
                    saved_diagnostic.get('removed_coordinates', '')
                ).split(';'))) == ASA_COMPONENT_GEPS
            )
            if not resume_is_current:
                print(
                    f'[{bundle_number}/{len(paths)}] STALE ASA checkpoint; recomputing',
                    flush=True,
                )

        if (
            resume_is_current
            and not OVERWRITE
        ):
            saved = json.loads(checkpoint.read_text())
            rows = saved['results']
            diagnostic = saved['diagnostic']
            nulls = {
                family: np.load(path, allow_pickle=False)['null']
                for family, path in expected_nulls.items()
            }
            print(f'[{bundle_number}/{len(paths)}] RESUME {stem}', flush=True)
        else:
            started = time.time()
            rows, diagnostic, nulls = evaluate_bundle(
                path, scope, bundle_number, output_directory,
                fold_number=fold_lookup.get(heldout),
            )
            for family, null in nulls.items():
                np.savez_compressed(expected_nulls[family], null=null)
            checkpoint.write_text(json.dumps({
                'bundle': str(path), 'results': rows, 'diagnostic': diagnostic,
            }, indent=2))
            print(
                f'[{bundle_number}/{len(paths)}] {stem} | '
                f'{(time.time() - started) / 60:.1f} min',
                flush=True,
            )

        all_rows.extend(rows)
        all_diagnostics.append(diagnostic)
        for family, null in nulls.items():
            null_lookup[(representation, family, heldout, context)] = null

    return pd.DataFrame(all_rows), pd.DataFrame(all_diagnostics), null_lookup

In [ ]:
crossfit_bundles = sorted(
    CROSSFIT_INPUT_DIR.glob('*/compositional_sensitivity_bundles/*__O_lambda.npz')
)
if not crossfit_bundles:
    raise FileNotFoundError('No cross-fitted compositional-sensitivity bundles were found.')

print(f'Cross-fitted bundles: {len(crossfit_bundles)}')
crossfit_rows, crossfit_diagnostics, crossfit_nulls = run_collection(
    crossfit_bundles, 'fully_crossfit', CROSSFIT_OUTPUT_DIR
)
crossfit_rows.to_csv(
    CROSSFIT_OUTPUT_DIR / 'crossfit_compositional_surrogate_by_dataset.csv',
    index=False,
)
crossfit_diagnostics.to_csv(
    CROSSFIT_OUTPUT_DIR / 'crossfit_projection_diagnostics.csv', index=False
)


def combine_crossfit(folds, null_lookup, representation, family):
    folds = folds.sort_values('heldout_dataset').reset_index(drop=True)
    standardized = []
    for fold in folds.itertuples(index=False):
        null = null_lookup[(representation, family, fold.heldout_dataset, fold.context)]
        null_mean = null.mean(axis=0)
        null_sd = np.maximum(null.std(axis=0, ddof=1), 1e-12)
        standardized.append((null - null_mean) / null_sd * DESIRED_SIGN)
    standardized = np.stack(standardized)

    rows = []
    for metric_number, metric in enumerate(METRICS):
        observed = np.mean(
            folds[f'z__{metric}'].to_numpy(float) * DESIRED_SIGN[metric_number]
        )
        null = standardized[:, :, metric_number].mean(axis=0)
        rows.append({
            'representation': representation,
            'surrogate_family': family,
            'metric': metric,
            'datasets': len(folds),
            'mean_oriented_z': observed,
            'surrogate_p': (1 + np.sum(null >= observed)) / (len(null) + 1),
            'datasets_in_expected_direction': int(np.sum(
                folds[f'contrast__{metric}'].to_numpy(float)
                * DESIRED_SIGN[metric_number] > 0
            )),
        })

    observed = folds.joint_oriented_z.mean()
    null = standardized.mean(axis=(0, 2))
    rows.append({
        'representation': representation,
        'surrogate_family': family,
        'metric': 'three_metric_joint',
        'datasets': len(folds),
        'mean_oriented_z': observed,
        'surrogate_p': (1 + np.sum(null >= observed)) / (len(null) + 1),
        'datasets_in_expected_direction': int(np.sum(folds.joint_oriented_z > 0)),
    })
    return rows


combined_rows = []
for (representation, family), folds in crossfit_rows.groupby(
    ['representation', 'surrogate_family']
):
    combined_rows.extend(
        combine_crossfit(folds.copy(), crossfit_nulls, representation, family)
    )
crossfit_combined = pd.DataFrame(combined_rows)
crossfit_combined.to_csv(
    CROSSFIT_OUTPUT_DIR / 'crossfit_compositional_surrogate_combined.csv',
    index=False,
)
display(crossfit_combined)

In [ ]:
unconstrained_path = CROSSFIT_INPUT_DIR / 'mean_random_field_combined.csv'
if unconstrained_path.exists():
    unconstrained = pd.read_csv(unconstrained_path)
    comparison = crossfit_combined.merge(
        unconstrained,
        on=['representation', 'surrogate_family', 'metric'],
        how='left',
        suffixes=('__constrained', '__unconstrained'),
    )
    comparison['conclusion_changed_at_0_05'] = (
        comparison.surrogate_p__constrained.lt(.05)
        != comparison.surrogate_p__unconstrained.lt(.05)
    )
    comparison.to_csv(
        CROSSFIT_OUTPUT_DIR / 'constrained_vs_unconstrained_crossfit.csv',
        index=False,
    )
    display(comparison)
else:
    comparison = pd.DataFrame()
    print('Existing unconstrained cross-fitted summary was not found.')

In [ ]:
display(pd.read_csv(OUTPUT_DIR/'mean_crossfit_predictive_field_combined.csv'))
print('Field and unconstrained surrogate tables:',OUTPUT_DIR)
print('CLR sensitivity tables:',CROSSFIT_OUTPUT_DIR)